# 02 / Persistent Memory with Vector Embeddings

**Deep Learning Indaba 2026 / Introduction to Agentic AI**

This is the heart of the tutorial. We give our griot **long-term memory that survives a
restart** and that it can **search by meaning**, so a saga can grow over many sittings and
the storyteller always recalls the right past event.

The recipe is the standard one used in production:

1. Turn each memory (a story event) into a **vector embedding**.
2. Store the vectors.
3. For a new moment, embed it and **retrieve the most similar** memories.
4. **Augment** the prompt with them, then continue the tale.

**No API key:** both the language model and the embedding model are open and run locally.

## 2.1 / Setup

In [ ]:
%pip install -q transformers accelerate sentence-transformers

In [ ]:
import os, json
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
import torch

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype="auto", device_map="auto")

def chat(messages, max_new_tokens=256, temperature=0.8):
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    gen = dict(max_new_tokens=max_new_tokens, pad_token_id=tok.eos_token_id)
    if temperature and temperature > 0:
        gen.update(do_sample=True, temperature=temperature, top_p=0.9)
    else:
        gen.update(do_sample=False)                  # greedy decoding when temperature is 0
    out = model.generate(**inputs, **gen)
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

embedder = SentenceTransformer("all-MiniLM-L6-v2")   # small open embedding model, no key
print("Ready on", model.device)

## 2.2 / What is an embedding?

An **embedding** turns text into a vector, a list of numbers that captures its *meaning*.
Texts with similar meaning get vectors pointing in similar directions, even with no shared
words.

In [ ]:
def embed(text):
    return embedder.encode(text)

v = embed("The hunter found a glowing stone by the river.")
print("Vector length:", len(v))            # 384 numbers for all-MiniLM-L6-v2
print("First 8 values:", np.round(v[:8], 3))

### Similarity is direction, not shared words

We compare two vectors with **cosine similarity** (1.0 means the same meaning, 0 means
unrelated). The two river sentences score high together; the unrelated one scores low.

In [ ]:
def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

a = embed("The hunter found a glowing stone by the river.")
b = embed("Near the water, Ado discovered a shining rock.")
d = embed("The market was crowded on the day of the festival.")
print(f"river stone  vs  shining rock by water : {cosine(a, b):.3f}   (similar)")
print(f"river stone  vs  crowded market        : {cosine(a, d):.3f}   (unrelated)")

## 2.3 / A tiny vector store

A *vector store* keeps texts with their embeddings and returns the most similar ones to a
query. Production systems use a dedicated vector database with a fast index (such as HNSW in
Redis), but the core idea fits in about 25 lines.

In [ ]:
class VectorMemory:
    """A minimal, persistent long-term memory backed by embeddings."""
    def __init__(self, path="saga_memory.json"):
        self.path = path
        self.texts, self.vectors = [], []
        self._load()
    def add(self, text):
        self.texts.append(text)
        self.vectors.append(np.array(embed(text), dtype=np.float32))
        self._save()
    def search(self, query, k=3):
        if not self.texts:
            return []
        q = embed(query)
        scores = [cosine(q, v) for v in self.vectors]
        return sorted(zip(scores, self.texts), reverse=True)[:k]
    def _save(self):
        json.dump({"texts": self.texts, "vectors": [v.tolist() for v in self.vectors]}, open(self.path, "w"))
    def _load(self):
        if os.path.exists(self.path):
            d = json.load(open(self.path))
            self.texts = d["texts"]
            self.vectors = [np.array(v, dtype=np.float32) for v in d["vectors"]]
            print(f"Loaded {len(self.texts)} memories from disk.")

memory = VectorMemory()
print("Memory ready. Items:", len(memory.texts))

## 2.4 / Recording the saga's events

Imagine these events built up over several sittings. We store them once; they now live on
disk in `saga_memory.json`.

In [ ]:
events = [
    "The heroine is Zola, a young weaver from the village by the great river.",
    "Zola found a talking drum that warns her of danger.",
    "By the river, the hunter Ado discovered a glowing stone that hums at night.",
    "A wise tortoise guards three riddles at the edge of the forest.",
    "The baobab tree is a safe shelter where elders once held council.",
    "Zola promised the drum she would never use it for revenge.",
]
for e in events:
    memory.add(e)
print("Stored", len(memory.texts), "events.")

## 2.5 / Retrieve, augment, respond

When we continue the tale, we **retrieve** only the relevant past events (not the whole
history), **inject** them into the prompt, and let the griot continue, recalling the right
detail by *meaning*.

In [ ]:
def continue_tale(prompt_text, k=3, show=True):
    hits = memory.search(prompt_text, k=k)
    recalled = "\n".join(f"- {t}" for _, t in hits)
    if show:
        print("Retrieved events:")
        for score, t in hits:
            print(f"  ({score:.2f}) {t}")
        print()
    reply = chat([
        {"role": "system", "content": "You are a griot continuing one consistent folktale. Use the recalled events so the story stays coherent. Reply in one short paragraph."},
        {"role": "user", "content": f"Recalled events:\n{recalled}\n\nNow continue: {prompt_text}"},
    ])
    return reply

print(continue_tale("Bring back the shiny thing the hunter found near the water."))

We never named the *glowing stone* or *Ado*; the griot retrieved them by meaning
("shiny thing near the water") and wove them back in. Try another:

In [ ]:
print(continue_tale("Remind the audience of the promise the heroine made."))

## 2.6 / The saga survives a restart

This is the difference from notebook `01`. The events are on disk. To prove it: in a fresh
kernel (Runtime > Restart), run only the setup cell, then the cell below, **without** adding
any events, and the griot still knows the whole saga.

In [ ]:
fresh = VectorMemory()                  # loads saga_memory.json
print("Events recovered from disk:", len(fresh.texts))
for score, t in fresh.search("what did the heroine promise?", k=2):
    print(f"  ({score:.2f}) {t}")

## 2.7 / From prototype to production

What you built is the shape of how real memory-backed agents work, just smaller:

| Your prototype | A real production system |
|---|---|
| `embed()` with all-MiniLM-L6-v2 | A managed or self-hosted embedding model |
| Python list and a cosine loop | A vector database with a fast index (such as HNSW) |
| `saga_memory.json` on disk | A store split into short-term (with a TTL) and permanent memory |
| Retrieve top-k events | A context step that fetches several sources in parallel |
| One saga, a few events | Thousands of items and many users |

The principles are identical: **embed, store, retrieve by similarity, augment the prompt.**
And because every piece here is open and runs locally, you can host the whole thing yourself,
on your own terms. That is what sovereign, locally-owned AI looks like in practice.

## Optional: chat with your griot (a simple interface)

For the live "talking to the griot" moment, a small chat box is friendlier than re-running a
cell, and you can invite the room to type. This is an **optional** extra: it needs one more
package (`gradio`). If anything misbehaves during the session, just skip this cell and keep
using the functions above.

In [ ]:
# Optional. If this install fails or gradio will not import, skip it and use the
# zero-dependency version further down. %pip installs into THIS kernel (use %pip, not !pip).
%pip install -q gradio

In [ ]:
import gradio as gr

# A short running memory of the current chat, on top of the persistent vector memory above.
CHAT_SYSTEM = "You are a griot, a warm oral storyteller. Continue one consistent tale, replying in a short paragraph."
session = []

def griot_reply(message, history):
    # recall relevant long-term events by meaning
    hits = memory.search(message, k=3)
    recalled = "\n".join(f"- {t}" for _, t in hits)
    system = CHAT_SYSTEM + (("\n\nEvents you recall:\n" + recalled) if recalled else "")
    msgs = [{"role": "system", "content": system}] + session[-6:] + [{"role": "user", "content": message}]
    reply = chat(msgs, max_new_tokens=256)
    session.extend([{"role": "user", "content": message},
                    {"role": "assistant", "content": reply}])
    return reply

# launch() shows the chat inline in Colab. Add share=True for a public link you can pass around the room.
gr.ChatInterface(griot_reply, title="Talk to the Griot").launch()

### No gradio? A zero-dependency chat

If `gradio` will not install (common on some local setups), this version needs nothing extra.
Run the cell, type a message, and enter `quit` to stop. It uses the same memory-backed agent.

In [ ]:
CHAT_SYSTEM = "You are a griot, a warm oral storyteller. Continue one consistent tale, replying in a short paragraph."
session = []
print("Talk to the griot (type 'quit' to stop).\n")
while True:
    message = input("You: ").strip()
    if message.lower() in {"quit", "exit", ""}:
        print("(ended)"); break
    hits = memory.search(message, k=3)
    recalled = "\n".join(f"- {t}" for _, t in hits)
    system = CHAT_SYSTEM + (("\n\nEvents you recall:\n" + recalled) if recalled else "")
    msgs = [{"role": "system", "content": system}] + session[-6:] + [{"role": "user", "content": message}]
    reply = chat(msgs, max_new_tokens=256)
    session += [{"role": "user", "content": message}, {"role": "assistant", "content": reply}]
    print("Griot:", reply, "\n")

## Recap of the whole tutorial

- **`00`**: an agent is a model plus instructions plus tools plus a perceive, reason, act loop.
- **`01`**: short-term memory is just resending history; it is bounded by the context window
  and costs tokens, so we compress it.
- **`02`**: persistent long-term memory uses **embeddings**: store events as vectors, retrieve
  the relevant ones by meaning, and augment the prompt. It survives restarts and scales.

You built all of it with open models, no API key, and nothing leaving the machine.